# Section 6: ETL Pipeline

ETL stands for:

- Extract
- Transform
- Load

It is one of the most important processes in Data Engineering.

In this section:

1. Data is extracted from the transformed CSV file.
2. Data is transformed by handling missing values and removing duplicates.
3. Clean data is loaded into a new output CSV file.

This pipeline simulates a real-world data engineering workflow.

In [ ]:
import pandas as pd

# ==========================================
# EXTRACT
# ==========================================

def extract():

    print("EXTRACT STAGE")
    print("-"*50)

    df = pd.read_csv(
        "campus_wifi_transformed.csv"
    )

    print(f"Rows Loaded: {df.shape[0]}")

    return df


# ==========================================
# TRANSFORM
# ==========================================

def transform(df):

    print("\nTRANSFORM STAGE")
    print("-"*50)

    df["Download_Speed_Mbps"] = df[
        "Download_Speed_Mbps"
    ].fillna(
        df["Download_Speed_Mbps"].median()
    )

    df["Signal_Strength_dBm"] = df[
        "Signal_Strength_dBm"
    ].fillna(
        df["Signal_Strength_dBm"].median()
    )

    before_rows = df.shape[0]

    df.drop_duplicates(inplace=True)

    after_rows = df.shape[0]

    print(f"Duplicates Removed: {before_rows - after_rows}")

    return df


# ==========================================
# LOAD
# ==========================================

def load(df):

    print("\nLOAD STAGE")
    print("-"*50)

    output_file = "etl_output.csv"

    df.to_csv(
        output_file,
        index=False
    )

    print(f"Output Saved: {output_file}")


print("ETL Functions Created Successfully")

ETL Functions Created Successfully


In [ ]:
# ==========================================
# RUN ETL PIPELINE
# ==========================================

etl_df = extract()

etl_df = transform(etl_df)

load(etl_df)

print("\nETL PIPELINE COMPLETED SUCCESSFULLY")

EXTRACT STAGE
--------------------------------------------------
Rows Loaded: 10000

TRANSFORM STAGE
--------------------------------------------------
Duplicates Removed: 0

LOAD STAGE
--------------------------------------------------
Output Saved: etl_output.csv

ETL PIPELINE COMPLETED SUCCESSFULLY


In [ ]:
final_df = pd.read_csv(
    "etl_output.csv"
)

print(final_df.shape)

display(final_df.head())

(10000, 17)


,Record_ID,Timestamp,Building,Floor,Zone,Connected_Users,Bandwidth_Usage_MB,Download_Speed_Mbps,Upload_Speed_Mbps,Latency_ms,Packet_Loss_Percent,Signal_Strength_dBm,Weather,Event_Occurring,Congestion_Status,Traffic_Level,Network_Health
0,1,2025-09-12 01:18:00,AMENITY,GROUND,Music Room,100,1137.17,95.00,50.00,73.00,3.39,-58.0,Rainy,Culturals,Medium,Moderate,Good
1,2,2025-03-02 18:18:00,MECH_BLOCK,GROUND,FABLAB,63,747.05,115.35,59.25,48.95,1.87,-57.0,Sunny,No_Event,Low,Light,Excellent
2,3,2025-05-19 05:21:00,MARIO,GROUND,Snack Corner,115,1323.06,86.75,46.25,82.75,2.81,-58.0,Sunny,No_Event,Medium,Moderate,Good
3,4,2025-04-19 20:10:00,AI_BLOCK,FLOOR3,Data Visualization Lab,94,1395.81,98.30,51.50,69.10,3.31,-44.0,Rainy,No_Event,Low,Light,Good
4,5,2025-12-06 22:12:00,MAIN_BLOCK,FLOOR2,IT Project Lab,57,828.78,118.65,60.75,45.05,3.57,-72.0,Sunny,Freshwarite,Low,Light,Excellent


## Observations

1. Data was successfully extracted from the transformed dataset.
2. Missing values were handled during the transformation stage.
3. Duplicate records were removed.
4. The cleaned data was loaded into a new output file.
5. ETL is a fundamental process used in real-world data engineering systems.

# Section 7: PySpark Medallion Architecture

The Medallion Architecture is a data design pattern used in modern data engineering.

It consists of three layers:

### Bronze Layer
Stores raw ingested data.

### Silver Layer
Stores cleaned and standardized data.

### Gold Layer
Stores business-ready datasets and KPIs.

In this project, PySpark is used to implement the Bronze, Silver, and Gold layers using Parquet format.

In [ ]:
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count

spark = SparkSession.builder \
    .appName("CampusConnectivityMedallion") \
    .getOrCreate()

print("Spark Session Created Successfully")

Spark Session Created Successfully


In [ ]:
# ==========================================
# BRONZE LAYER
# Raw Data
# ==========================================

bronze_df = spark.read.csv(
    "etl_output.csv",
    header=True,
    inferSchema=True
)

print("="*60)
print("BRONZE LAYER")
print("="*60)

bronze_df.show(5)

bronze_df.write.mode("overwrite").parquet(
    "bronze_layer"
)

print("Bronze Layer Saved Successfully")

BRONZE LAYER
+---------+-------------------+----------+------+--------------------+---------------+------------------+-------------------+-----------------+----------+-------------------+-------------------+-------+---------------+-----------------+-------------+--------------+
|Record_ID|          Timestamp|  Building| Floor|                Zone|Connected_Users|Bandwidth_Usage_MB|Download_Speed_Mbps|Upload_Speed_Mbps|Latency_ms|Packet_Loss_Percent|Signal_Strength_dBm|Weather|Event_Occurring|Congestion_Status|Traffic_Level|Network_Health|
+---------+-------------------+----------+------+--------------------+---------------+------------------+-------------------+-----------------+----------+-------------------+-------------------+-------+---------------+-----------------+-------------+--------------+
|        1|2025-09-12 01:18:00|   AMENITY|GROUND|          Music Room|            100|           1137.17|               95.0|             50.0|      73.0|               3.39|              -

In [ ]:
# ==========================================
# SILVER LAYER
# Cleaned Data
# ==========================================

silver_df = bronze_df.dropDuplicates()

silver_df = silver_df.na.fill({
    "Download_Speed_Mbps": 0,
    "Signal_Strength_dBm": -60
})

print("="*60)
print("SILVER LAYER")
print("="*60)

silver_df.show(5)

silver_df.write.mode("overwrite").parquet(
    "silver_layer"
)

print("Silver Layer Saved Successfully")

SILVER LAYER
+---------+-------------------+----------+------+------------+---------------+------------------+-------------------+-----------------+----------+-------------------+-------------------+-------+---------------+-----------------+-------------+--------------+
|Record_ID|          Timestamp|  Building| Floor|        Zone|Connected_Users|Bandwidth_Usage_MB|Download_Speed_Mbps|Upload_Speed_Mbps|Latency_ms|Packet_Loss_Percent|Signal_Strength_dBm|Weather|Event_Occurring|Congestion_Status|Traffic_Level|Network_Health|
+---------+-------------------+----------+------+------------+---------------+------------------+-------------------+-----------------+----------+-------------------+-------------------+-------+---------------+-----------------+-------------+--------------+
|       66|2025-12-05 12:22:00|  AI_BLOCK|FLOOR3|   Cyber Lab|            146|           1408.44|               69.7|             38.5|     102.9|               4.56|              -61.0| Cloudy|       No_Event|   

In [ ]:
# ==========================================
# GOLD LAYER
# Business KPIs
# ==========================================

gold_df = silver_df.groupBy(
    "Building"
).agg(
    avg("Download_Speed_Mbps").alias("Avg_Download_Speed"),
    avg("Latency_ms").alias("Avg_Latency"),
    count("*").alias("Total_Records")
)

print("="*60)
print("GOLD LAYER")
print("="*60)

gold_df.show()

gold_df.write.mode("overwrite").parquet(
    "gold_layer"
)

print("Gold Layer Saved Successfully")

GOLD LAYER
+------------+------------------+-----------------+-------------+
|    Building|Avg_Download_Speed|      Avg_Latency|Total_Records|
+------------+------------------+-----------------+-------------+
| HOSTEL_BOYS| 89.25830388692579|82.60759717314485|          283|
|     AMENITY| 80.97604327666146|95.79822256568775|          647|
|  MAIN_BLOCK|  89.9811406460295|80.37330080753712|         2972|
|    AI_BLOCK| 85.47809569996738|86.50835758163596|         3093|
|  MECH_BLOCK| 92.86458333333316|75.63385416666671|         2112|
|HOSTEL_GIRLS| 86.51513605442169| 84.4391156462585|          294|
|    BUS_AREA|106.42047781569968|59.15921501706477|          293|
|       MARIO| 95.37369281045748|73.21029411764702|          306|
+------------+------------------+-----------------+-------------+

Gold Layer Saved Successfully


In [ ]:
print("="*60)
print("BRONZE COUNT")
print("="*60)

print(bronze_df.count())

print("="*60)
print("SILVER COUNT")
print("="*60)

print(silver_df.count())

print("="*60)
print("GOLD COUNT")
print("="*60)

print(gold_df.count())

BRONZE COUNT
10000
SILVER COUNT
10000
GOLD COUNT
8


## Observations

### Bronze Layer
The raw ETL dataset was ingested and stored in Parquet format.

### Silver Layer
Duplicate records were removed and missing values were standardized.

### Gold Layer
Business-ready KPIs were generated including:

- Average Download Speed by Building
- Average Latency by Building
- Total Records by Building

All three layers were stored in Parquet format, demonstrating the Medallion Architecture using PySpark.